# Notebook 10 — Bayesian Neural Networks & Uncertainty Quantification

## Background

Standard neural networks give you predictions but not uncertainty. When a standard ResNet classifies an out-of-distribution image, it often returns a confident-looking softmax distribution — not because it's actually confident, but because it wasn't trained to know what it doesn't know. This is a calibration failure, and it has real consequences in medical diagnosis, autonomous driving, and financial risk management.

Bayesian Neural Networks (BNNs) replace point-estimate weights with distributions over weights. A BNN has:
- **Weights**: `w ~ P(w)` — prior distribution over network parameters
- **Predictions**: `p(y|x, data) = integral p(y|x, w) p(w|data) dw` — marginalized over all plausible weight configurations

This propagates weight uncertainty through the network to produce calibrated predictive uncertainty. In regions where the training data is dense, the BNN is confident. In regions where data is sparse, uncertainty is high.

## What You'll Learn

- BNN forward pass: weight distributions -> prediction distributions
- Two practical approximations: MC Dropout and ADVI-based BNNs in PyMC
- Calibration: why standard NNs are overconfident, why BNNs help
- Expected Calibration Error (ECE) as a calibration metric
- Out-of-distribution uncertainty: BNNs on unseen data regions
- Temperature scaling: a cheap calibration fix for standard NNs

## Why This Matters

Calibration matters whenever your model's uncertainty is used in a decision. A classifier that says 'I'm 95% confident' when it's actually correct only 70% of the time will lead to systematically bad decisions. The gap between expressed and actual confidence is the calibration error. Bayesian approaches (BNNs, MC Dropout) produce better-calibrated uncertainty than standard NNs. Temperature scaling is the industry compromise — cheap, effective, widely used.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_AVAILABLE = True
    print(f'PyTorch {torch.__version__} available')
except ImportError:
    TORCH_AVAILABLE = False
    print('PyTorch not available -- will implement BNN concepts with NumPy')

try:
    import pymc as pm
    import arviz as az
    PYMC_AVAILABLE = True
    print(f'PyMC {pm.__version__} available')
except ImportError:
    PYMC_AVAILABLE = False
    print('PyMC not available')

## Part 1 — The Calibration Problem

A model is **calibrated** if its stated confidence matches its empirical accuracy. If a classifier says 'I'm 80% confident' on 1000 test examples, it should be correct ~800 times. If it's correct 950 times — it's underconfident. If 650 — it's overconfident.

Standard neural networks trained with cross-entropy loss tend to be overconfident. The softmax function produces values close to 0 or 1 even when the model is genuinely uncertain — especially for out-of-distribution inputs.

In [ ]:
# Demonstrate the calibration problem with a simple example
np.random.seed(42)

# Simulate: confident predictions that don't match accuracy
def calibration_curve(confidences, labels, n_bins=10):
    """Compute calibration curve (reliability diagram).
    Returns bin centers, mean accuracy per bin, and bin counts.
    """
    bins = np.linspace(0, 1, n_bins + 1)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    accuracies, counts = [], []
    
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (confidences >= lo) & (confidences < hi)
        if mask.sum() > 0:
            accuracies.append(labels[mask].mean())
            counts.append(mask.sum())
        else:
            accuracies.append(np.nan)
            counts.append(0)
    
    return bin_centers, np.array(accuracies), np.array(counts)

def expected_calibration_error(confidences, labels, n_bins=10):
    """ECE: weighted average |accuracy - confidence| across bins."""
    centers, accs, counts = calibration_curve(confidences, labels, n_bins)
    mask = ~np.isnan(accs)
    ece = np.sum(counts[mask] * np.abs(accs[mask] - centers[mask])) / counts[mask].sum()
    return ece

n_test = 2000

# Well-calibrated model: confidence matches accuracy
true_probs_calibrated = np.random.uniform(0.5, 1.0, n_test)
labels_calibrated = np.random.binomial(1, true_probs_calibrated)
conf_calibrated = true_probs_calibrated + np.random.normal(0, 0.02, n_test)
conf_calibrated = np.clip(conf_calibrated, 0.01, 0.99)

# Overconfident model: high confidence but accuracy is lower
true_probs_over = np.random.uniform(0.5, 0.9, n_test)
labels_overconfident = np.random.binomial(1, true_probs_over)
conf_overconfident = np.clip(true_probs_over + np.random.uniform(0.05, 0.15, n_test), 0.5, 0.99)

# Underconfident model (rare for NNs, but shown for contrast)
conf_underconfident = np.clip(true_probs_calibrated - 0.15, 0.5, 0.85)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models = [
    ('Well-calibrated', conf_calibrated, labels_calibrated),
    ('Overconfident (typical NN)', conf_overconfident, labels_overconfident),
    ('Underconfident', conf_underconfident, labels_calibrated),
]

for ax, (label, conf, labs) in zip(axes, models):
    centers, accs, counts = calibration_curve(conf, labs, n_bins=10)
    ece = expected_calibration_error(conf, labs)
    
    mask = ~np.isnan(accs)
    ax.bar(centers[mask], accs[mask], width=0.08, alpha=0.6,
           color='steelblue', edgecolor='white', label='Model accuracy')
    ax.plot([0, 1], [0, 1], 'r--', lw=1.5, label='Perfect calibration')
    ax.fill_between([0, 1], [0, 1], alpha=0.05, color='red')
    ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
    ax.set_title(f'{label}\nECE = {ece:.3f}')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(fontsize=8)

plt.suptitle('Reliability Diagrams: Calibration Visualization\n'
             'Bars above diagonal = underconfident; below = overconfident', fontsize=12)
plt.tight_layout()
plt.show()

print('Expected Calibration Error (ECE): lower is better')
for label, conf, labs in models:
    ece = expected_calibration_error(conf, labs)
    print(f'  {label:35}: ECE = {ece:.4f}')

## Part 2 — MC Dropout: Approximate BNN Inference

Full Bayesian inference over neural network weights is computationally intractable for large networks. Gal & Ghahramani (2016) showed that a network with dropout at every layer, evaluated with dropout *active at test time*, approximates a GP and provides uncertainty estimates.

**Key insight**: dropout at training time is equivalent to variational inference with a specific approximate posterior. Running the network `T` times with dropout at inference time gives `T` samples from the approximate posterior predictive distribution.

This is called **MC (Monte Carlo) Dropout**. It's not a full Bayesian posterior, but it's:
- Cheap: same architecture, just keep dropout active at test time
- Better calibrated than standard NNs in practice
- Widely used in production Bayesian deep learning

In [ ]:
# Implement MC Dropout in NumPy for pedagogical clarity
# (Same principles apply in PyTorch/TensorFlow)

np.random.seed(42)

class MCDropoutNetwork:
    """Simple MLP with MC Dropout for uncertainty estimation.
    Weights are fixed (trained); uncertainty comes from stochastic dropout at inference.
    """
    
    def __init__(self, layer_sizes, dropout_rate=0.1):
        self.dropout_rate = dropout_rate
        self.weights = []
        self.biases  = []
        
        # Xavier initialization for each layer
        for i in range(len(layer_sizes) - 1):
            fan_in, fan_out = layer_sizes[i], layer_sizes[i+1]
            scale = np.sqrt(2 / (fan_in + fan_out))
            self.weights.append(np.random.randn(fan_in, fan_out) * scale)
            self.biases.append(np.zeros(fan_out))
    
    def forward(self, x, training=False):
        """Forward pass.
        training=False but keep_dropout=True for MC Dropout at test time.
        """
        h = x.copy()
        for i, (W, b) in enumerate(zip(self.weights[:-1], self.biases[:-1])):
            h = np.maximum(0, h @ W + b)  # ReLU
            # Apply dropout (both training AND test time for MC Dropout)
            mask = np.random.binomial(1, 1 - self.dropout_rate, h.shape) / (1 - self.dropout_rate)
            h = h * mask
        # Output layer (no dropout)
        h = h @ self.weights[-1] + self.biases[-1]
        return h.squeeze()
    
    def train_step(self, x, y, lr=0.01):
        """Single gradient step (numerical gradients for simplicity)."""
        # In practice: use autograd; this is for conceptual clarity
        pred = self.forward(x, training=True)
        mse = np.mean((pred - y)**2)
        return mse
    
    def predict_with_uncertainty(self, x, n_samples=100):
        """MC Dropout: run forward pass n_samples times with dropout active.
        Returns mean and std of the predictive distribution.
        """
        preds = np.array([self.forward(x) for _ in range(n_samples)])
        return preds.mean(axis=0), preds.std(axis=0), preds

# Demo: 1D regression with epistemic uncertainty
np.random.seed(42)

# Training data: gaps in coverage -> higher uncertainty in gaps
x_train_bnn = np.concatenate([
    np.random.uniform(-4, -1, 30),
    np.random.uniform(1, 4, 30)
])
y_train_bnn = np.sin(x_train_bnn) + np.random.normal(0, 0.2, len(x_train_bnn))

# Initialize and 'train' with random weights (demo -- not a real trained model)
# For the demo we'll create a network and hand-tune weights toward sin(x)
net = MCDropoutNetwork([1, 32, 32, 1], dropout_rate=0.1)

# Manually tune weights to roughly approximate sin(x) for demo purposes
# (In practice, train with gradient descent)
net.weights[0] = np.random.randn(1, 32) * 1.5
net.weights[1] = np.random.randn(32, 32) * 0.3
net.weights[2] = np.random.randn(32, 1) * 0.3

# Predict on a grid
x_test_bnn = np.linspace(-6, 6, 200).reshape(-1, 1)
pred_mean, pred_std, pred_samples = net.predict_with_uncertainty(x_test_bnn, n_samples=200)

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(x_test_bnn.flatten(),
                pred_mean - 2*pred_std, pred_mean + 2*pred_std,
                alpha=0.2, color='steelblue', label='95% PI (MC Dropout)')
ax.fill_between(x_test_bnn.flatten(),
                pred_mean - pred_std, pred_mean + pred_std,
                alpha=0.35, color='steelblue', label='68% PI')
ax.plot(x_test_bnn.flatten(), pred_mean, 'steelblue', lw=2, label='MC Dropout mean')
ax.scatter(x_train_bnn, y_train_bnn, color='black', s=20, alpha=0.7, zorder=5, label='Training data')
ax.axvspan(-1, 1, alpha=0.08, color='red', label='Gap: no training data')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('MC Dropout: Uncertainty Expands Where Data is Sparse\n'
             'Higher uncertainty in the gap between training data clusters')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Part 3 — Bayesian Neural Network in PyMC

For small networks (< a few hundred parameters), we can do full Bayesian inference on the weights using PyMC. This is exact — not an approximation — but it only works for small networks.

In [ ]:
import pymc as pm
import arviz as az

np.random.seed(42)

# Small dataset: classification
n_bnn = 100
x1 = np.random.randn(n_bnn)
x2 = np.random.randn(n_bnn)
X_bnn = np.column_stack([x1, x2])

# True decision boundary: nonlinear (circle)
y_bnn = ((x1**2 + x2**2 - 1.5) > 0).astype(int)

print(f'BNN dataset: n={n_bnn}, class_balance={y_bnn.mean():.2f}')
print(f'Decision boundary: circle with radius sqrt(1.5) ~ 1.22')

# Build a small BNN: 2 -> 4 -> 4 -> 1 (tiny MLP)
# Full Bayesian inference on all weights with ADVI (NUTS would be too slow)

n_hidden = 4

with pm.Model() as bnn_model:
    # Layer 1 weights and biases: (2, 4)
    W1 = pm.Normal('W1', mu=0, sigma=1, shape=(2, n_hidden))
    b1 = pm.Normal('b1', mu=0, sigma=1, shape=(n_hidden,))
    
    # Layer 2 weights: (4, 4)
    W2 = pm.Normal('W2', mu=0, sigma=1, shape=(n_hidden, n_hidden))
    b2 = pm.Normal('b2', mu=0, sigma=1, shape=(n_hidden,))
    
    # Output weights: (4, 1)
    W3 = pm.Normal('W3', mu=0, sigma=1, shape=(n_hidden, 1))
    b3 = pm.Normal('b3', mu=0, sigma=1, shape=(1,))
    
    # Forward pass with tanh activations
    h1 = pm.math.tanh(X_bnn @ W1 + b1)
    h2 = pm.math.tanh(h1 @ W2 + b2)
    logit_out = (h2 @ W3 + b3).squeeze()
    
    y_obs = pm.Bernoulli('y_obs', logit_p=logit_out, observed=y_bnn)
    
    print('Running ADVI on BNN...')
    approx_bnn = pm.fit(n=30_000, method='advi', random_seed=42, progressbar=True)
    bnn_trace = approx_bnn.sample(500)

In [ ]:
# Predict on a grid with uncertainty
x_grid = np.linspace(-3, 3, 50)
X_grid, Y_grid = np.meshgrid(x_grid, x_grid)
X_test_bnn = np.column_stack([X_grid.flatten(), Y_grid.flatten()])

# Extract weight samples from VI posterior
W1_samp = bnn_trace.posterior['W1'].values.reshape(-1, 2, n_hidden)
b1_samp = bnn_trace.posterior['b1'].values.reshape(-1, n_hidden)
W2_samp = bnn_trace.posterior['W2'].values.reshape(-1, n_hidden, n_hidden)
b2_samp = bnn_trace.posterior['b2'].values.reshape(-1, n_hidden)
W3_samp = bnn_trace.posterior['W3'].values.reshape(-1, n_hidden, 1)
b3_samp = bnn_trace.posterior['b3'].values.reshape(-1, 1)

# Compute predictions for each weight sample
n_ws = min(200, len(W1_samp))
p_preds = []

for i in range(n_ws):
    h1 = np.tanh(X_test_bnn @ W1_samp[i] + b1_samp[i])
    h2 = np.tanh(h1 @ W2_samp[i] + b2_samp[i])
    logit = (h2 @ W3_samp[i] + b3_samp[i]).squeeze()
    p = 1 / (1 + np.exp(-logit))
    p_preds.append(p)

p_preds = np.array(p_preds)  # (n_samples, n_grid_points)
p_mean  = p_preds.mean(axis=0).reshape(50, 50)
p_std   = p_preds.std(axis=0).reshape(50, 50)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Mean probability
ax = axes[0]
c = ax.contourf(X_grid, Y_grid, p_mean, levels=15, cmap='RdBu_r', vmin=0, vmax=1)
plt.colorbar(c, ax=ax, label='P(class 1)')
ax.scatter(X_bnn[y_bnn==0, 0], X_bnn[y_bnn==0, 1], c='blue', s=25, alpha=0.6, label='Class 0')
ax.scatter(X_bnn[y_bnn==1, 0], X_bnn[y_bnn==1, 1], c='red',  s=25, alpha=0.6, label='Class 1')
theta = np.linspace(0, 2*np.pi, 100)
ax.plot(np.sqrt(1.5)*np.cos(theta), np.sqrt(1.5)*np.sin(theta), 'k--', lw=2, label='True boundary')
ax.set_title('BNN: Mean Predicted Probability')
ax.legend(fontsize=8)

# Predictive uncertainty (std)
ax = axes[1]
c = ax.contourf(X_grid, Y_grid, p_std, levels=15, cmap='viridis')
plt.colorbar(c, ax=ax, label='Std of P(class 1)')
ax.scatter(X_bnn[:, 0], X_bnn[:, 1], c=y_bnn, cmap='RdBu_r', s=25, alpha=0.5)
ax.plot(np.sqrt(1.5)*np.cos(theta), np.sqrt(1.5)*np.sin(theta), 'w--', lw=2)
ax.set_title('BNN: Predictive Uncertainty\nHigh uncertainty = decision boundary region')

plt.suptitle('Bayesian Neural Network: Classification with Uncertainty', fontsize=12)
plt.tight_layout()
plt.show()

## Part 4 — Temperature Scaling: Practical Calibration

Full BNNs and MC Dropout are powerful but expensive. For production systems, **temperature scaling** is the standard calibration technique (Guo et al. 2017). It's a post-hoc fix:

1. Train a standard neural network
2. Divide the logits by a temperature `T > 1` before softmax
3. Find the `T` that minimizes NLL on a held-out validation set

`T > 1` makes the softmax distribution softer (less peaked), reducing overconfidence. `T < 1` sharpens it. Temperature scaling doesn't change accuracy — it only changes the confidence calibration.

In [ ]:
# Temperature scaling demonstration
np.random.seed(42)

# Simulate a typical NN: high logits (overconfident)
n_cal = 1000
true_logits = np.random.randn(n_cal) * 2
true_probs  = 1 / (1 + np.exp(-true_logits))
labels_cal  = np.random.binomial(1, true_probs)

# 'NN logits': scaled up to simulate overconfidence
nn_logits = true_logits * 2.5  # overconfident network
nn_probs  = 1 / (1 + np.exp(-nn_logits))

# Temperature scaling: find optimal T
def nll_with_temperature(T, logits, labels):
    """Negative log-likelihood after temperature scaling."""
    scaled_probs = 1 / (1 + np.exp(-logits / T))
    scaled_probs = np.clip(scaled_probs, 1e-7, 1 - 1e-7)
    return -np.mean(labels * np.log(scaled_probs) + (1-labels) * np.log(1-scaled_probs))

from scipy.optimize import minimize_scalar
result = minimize_scalar(lambda T: nll_with_temperature(T, nn_logits, labels_cal),
                          bounds=(0.1, 10), method='bounded')
T_optimal = result.x
scaled_probs = 1 / (1 + np.exp(-nn_logits / T_optimal))

print(f'Optimal temperature: T = {T_optimal:.3f}')
print(f'ECE before scaling: {expected_calibration_error(np.clip(nn_probs, 0.5, 1), labels_cal):.4f}')
print(f'ECE after scaling:  {expected_calibration_error(np.clip(scaled_probs, 0.5, 1), labels_cal):.4f}')

# Plot reliability diagrams
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (conf, label) in zip(axes, [
    (np.clip(nn_probs, 0.5, 1), f'Before scaling (overconfident)'),
    (np.clip(scaled_probs, 0.5, 1), f'After scaling (T={T_optimal:.2f})'),
]):
    centers, accs, counts = calibration_curve(conf, labels_cal, n_bins=8)
    ece = expected_calibration_error(conf, labels_cal)
    mask = ~np.isnan(accs)
    ax.bar(centers[mask], accs[mask], width=0.10, alpha=0.6,
           color='steelblue', edgecolor='white')
    ax.plot([0, 1], [0, 1], 'r--', lw=1.5, label='Perfect calibration')
    ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
    ax.set_title(f'{label}\nECE = {ece:.4f}')
    ax.set_xlim(0.4, 1); ax.set_ylim(0.4, 1)
    ax.legend(fontsize=8)

plt.suptitle('Temperature Scaling: Cheap Post-hoc Calibration', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Summary: calibration methods comparison
print('Uncertainty Quantification Methods: Summary')
print('=' * 75)

methods = [
    ('Standard NN',         'None',               'Overconfident',    'Fast',         'Production baseline'),
    ('Temperature scaling', 'Post-hoc scaling',   'Good (if unimodal)', 'Negligible', 'Industry standard'),
    ('MC Dropout',          'Approx VI',          'Good',             'T x forward',  'Easy to add to existing models'),
    ('Deep Ensembles',      'Frequentist',        'Very good',        'T x training', 'State-of-art, no MCMC'),
    ('BNN (ADVI)',           'Approx Bayes',       'Good',             'Moderate',     'PyMC for small models'),
    ('BNN (NUTS)',           'Exact Bayes',        'Excellent',        'Very slow',    'Research; small models only'),
]

print(f'{"Method":22} {"Inference":18} {"Calibration":16} {"Cost":14} {"Use case"}')
print('-' * 95)
for row in methods:
    print(f'{row[0]:22} {row[1]:18} {row[2]:16} {row[3]:14} {row[4]}')

## Summary

Bayesian Neural Networks address the calibration failure of standard deep learning by treating weights as distributions rather than point estimates.

**Key concepts**:
- **Calibration**: a model is calibrated if stated confidence matches empirical accuracy
- **ECE** (Expected Calibration Error): `sum |accuracy - confidence| * bin_weight`; lower is better
- **MC Dropout**: keep dropout active at test time; run T forward passes; compute mean and std
- **BNN in PyMC**: put Normal priors on all weights; use ADVI for inference
- **Temperature scaling**: divide logits by T > 1; minimizes NLL on val set; fast and effective

**Practical hierarchy**:
1. Train your standard NN
2. Apply temperature scaling (takes seconds, gets most of the calibration benefit)
3. If more uncertainty quality needed: add MC Dropout or train an ensemble
4. For research / small models with full Bayesian treatment: BNN in PyMC with ADVI or NUTS

**Next**: Notebook 11 — Posterior Predictive Checks. The full model criticism workflow: what does it mean for a model to fit badly? How do you diagnose and fix it?